# Deep Learning: A Comprehensive Guide

## Chapter 9 — Language Models

### Hands-On Exploration: Same Model, Different Tasks

### `hands_on_ch9.ipynb`

A single pre-trained model can be adapted to remarkably diverse tasks. This exploration builds direct intuition for what changes during fine-tuning and what stays the same — and confronts you with the distribution shift problem that makes deploying pre-trained models in novel domains risky.

## Learning Objectives

By the end of this notebook, you should be able to:

- Observe how the same base architecture, fine-tuned differently, produces different outputs on the same input
- Diagnose distribution shift by testing a model outside its training domain
- Compare how prompt structure changes a generative model's output

## Prerequisites

- Python 3.9+, `numpy`, `scikit-learn`
- **Optional:** `transformers` (for the real pretrained-encoder version of this activity) — install with `pip install transformers`
- No GPU required

## About the Models in This Notebook

The chapter's activity uses three real fine-tuned BERT variants (sentiment, NER, entailment) and a small GPT-style model, all pre-loaded via HuggingFace Transformers. This notebook **first attempts** to load real HuggingFace pipelines — if you run it in Colab with internet access, you get the genuine chapter activity. If no internet access is available, it falls back to a **simplified, fully offline version**: instead of three different NLP tasks, we train the *same* sentiment classifier architecture on three different domains of synthetic text (movie reviews, financial news, medical notes) using a shared TF-IDF representation as a stand-in for "the same base encoder." This keeps the core lesson — distribution shift — fully intact and demonstrable offline, even though it simplifies away from the chapter's three-distinct-tasks framing. The prompting section falls back to a small Markov-chain text generator, which is far cruder than a real GPT-style model but is still visibly sensitive to prompt phrasing.

## Setup — Imports and Reproducibility

In [1]:
import numpy as np

SEED = 42
np.random.seed(SEED)

HF_AVAILABLE = False
try:
    from transformers import pipeline
    sentiment_pipe = pipeline("sentiment-analysis")
    ner_pipe = pipeline("ner", grouped_entities=True)
    generator_pipe = pipeline("text-generation", model="gpt2")
    HF_AVAILABLE = True
    print("\u2713 Loaded real HuggingFace pipelines. Running the genuine chapter activity.")
except Exception as e:
    print(f"Could not load HuggingFace pipelines ({type(e).__name__}: no internet access, or")
    print("`transformers` not installed). Falling back to a simplified offline version.")
    print("Run this notebook in Google Colab with `pip install transformers` for the genuine activity.")


Could not load HuggingFace pipelines (ModuleNotFoundError: no internet access, or
`transformers` not installed). Falling back to a simplified offline version.
Run this notebook in Google Colab with `pip install transformers` for the genuine activity.


---
## If HuggingFace Loaded — The Genuine Chapter Activity

In [2]:
if HF_AVAILABLE:
    test_sentences = [
        "Dr. Sarah Chen at Stanford Medical Center reported that the treatment showed promising early results.",
        "The company's quarterly earnings disappointed investors, sending shares down 12% in after-hours trading.",
        "I found the film's third act genuinely moving -- the performances carried the weak script.",
    ]

    for sent in test_sentences:
        print(f"\nInput: {sent}")
        print(f"  Sentiment model: {sentiment_pipe(sent)}")
        print(f"  NER model: {ner_pipe(sent)}")

    print("\n--- Part 2: Distribution Shift ---")
    medical_sentences = [
        "The patient responded poorly to the initial treatment regimen.",
        "Recovery was slow but ultimately the outcome was favorable.",
        "Side effects were mild and the patient tolerated the procedure well.",
        "The intervention was contraindicated given the patient's prior history.",
    ]
    for sent in medical_sentences:
        print(f"{sent}\n  -> {sentiment_pipe(sent)}")

    print("\n--- Part 3: Prompt Structure ---")
    prompts = [
        "What causes thunderstorms?",
        "Explain briefly: What causes thunderstorms?",
        "Q: What causes rain?\nA: Rain forms when water vapor condenses.\nQ: What causes lightning?\nA: Lightning forms from electrical charge buildup in clouds.\nQ: What causes thunderstorms?\nA:",
    ]
    for p in prompts:
        out = generator_pipe(p, max_new_tokens=30, num_return_sequences=1)
        print(f"\nPrompt: {p}\nOutput: {out[0]['generated_text']}")


**Answer in this cell (HF mode):** For each input, which model's output seems most sensible given the task it was trained for? For which inputs does any model produce something that surprises you? Does the sentiment model's classification remain sensible on medical text? Do the three prompt formats differ in length, style, or accuracy?

---
## If HuggingFace Did Not Load — The Fallback Activity

### Part 1 (Fallback) — Same Architecture, Three Domains

We train three sentiment classifiers with **identical architecture** (TF-IDF + logistic regression) but on three different domains of synthetic training text, standing in for "the same base model, fine-tuned differently." 

In [3]:
if not HF_AVAILABLE:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression

    # Synthetic domain-specific training sentences (small, hand-built vocabularies per domain)
    movie_pos = ["a moving and beautifully acted film", "brilliant performances carried the story",
                 "a genuinely touching and well crafted movie", "the acting was superb and heartfelt",
                 "an emotionally resonant and powerful film", "wonderful direction and a gripping story"]
    movie_neg = ["a dull and poorly written film", "the acting was wooden and unconvincing",
                 "a disappointing and forgettable movie", "weak script and flat performances",
                 "boring pacing ruined an interesting premise", "a tedious and uninspired film"]

    finance_pos = ["earnings beat expectations this quarter", "shares rallied on strong guidance",
                   "revenue growth exceeded analyst forecasts", "the company posted record profits",
                   "investors cheered the strong quarterly report", "margins improved significantly this year"]
    finance_neg = ["earnings disappointed investors this quarter", "shares fell sharply after the report",
                   "revenue missed analyst expectations", "the company posted a significant loss",
                   "investors were spooked by weak guidance", "margins declined sharply this year"]

    medical_pos = ["the patient recovered fully after treatment", "the procedure was successful and well tolerated",
                   "symptoms resolved completely within days", "the treatment was effective with no complications",
                   "the patient responded well to therapy", "recovery was smooth and complete"]
    medical_neg = ["the patient responded poorly to treatment", "the procedure resulted in complications",
                   "symptoms persisted despite intervention", "the treatment was ineffective and discontinued",
                   "the patient did not tolerate the therapy well", "recovery was slow and incomplete"]

    def train_domain_classifier(pos, neg):
        texts = pos + neg
        labels = [1] * len(pos) + [0] * len(neg)
        vec = TfidfVectorizer()
        X = vec.fit_transform(texts)
        clf = LogisticRegression()
        clf.fit(X, labels)
        return vec, clf

    movie_vec, movie_clf = train_domain_classifier(movie_pos, movie_neg)
    finance_vec, finance_clf = train_domain_classifier(finance_pos, finance_neg)
    medical_vec, medical_clf = train_domain_classifier(medical_pos, medical_neg)

    print("Trained three domain-specific sentiment classifiers (movie, finance, medical),")
    print("all with the identical architecture: TF-IDF + logistic regression.")


Trained three domain-specific sentiment classifiers (movie, finance, medical),
all with the identical architecture: TF-IDF + logistic regression.


In [4]:
if not HF_AVAILABLE:
    test_sentences = [
        ("Movie-domain", "the performances carried the weak script", movie_vec, movie_clf),
        ("Finance-domain", "shares dropped after disappointing guidance", finance_vec, finance_clf),
        ("Medical-domain", "the patient tolerated the procedure well", medical_vec, medical_clf),
    ]
    for domain_name, sent, vec, clf in test_sentences:
        pred = clf.predict(vec.transform([sent]))[0]
        prob = clf.predict_proba(vec.transform([sent]))[0].max()
        print(f"{domain_name} classifier on \"{sent}\": {'positive' if pred else 'negative'} (confidence {prob:.2f})")


Movie-domain classifier on "the performances carried the weak script": negative (confidence 0.50)
Finance-domain classifier on "shares dropped after disappointing guidance": negative (confidence 0.53)
Medical-domain classifier on "the patient tolerated the procedure well": positive (confidence 0.53)


**Answer in this cell:** Does each domain-specific classifier behave sensibly on its own domain's test sentence? What do you predict will happen if you feed the *movie* classifier a *medical* sentence? Test that prediction in Part 2 below.

### Part 2 (Fallback) — Distribution Shift in Practice

Take the **movie-domain classifier** — trained only on movie-review-style language — and feed it language from a medical context.

In [5]:
if not HF_AVAILABLE:
    medical_sentences = [
        "The patient responded poorly to the initial treatment regimen.",
        "Recovery was slow but ultimately the outcome was favorable.",
        "Side effects were mild and the patient tolerated the procedure well.",
        "The intervention was contraindicated given the patient's prior history.",
    ]

    print("Movie-domain classifier applied to MEDICAL text (out of its training distribution):\n")
    for sent in medical_sentences:
        vec_input = movie_vec.transform([sent])
        pred = movie_clf.predict(vec_input)[0]
        prob = movie_clf.predict_proba(vec_input)[0].max()
        n_known_words = int((vec_input.toarray() > 0).sum())
        print(f"\"{sent}\"")
        print(f"  -> predicted: {'positive' if pred else 'negative'} (confidence {prob:.2f}, "
              f"recognized {n_known_words} vocabulary term(s) from movie-review training data)\n")


Movie-domain classifier applied to MEDICAL text (out of its training distribution):

"The patient responded poorly to the initial treatment regimen."
  -> predicted: negative (confidence 0.50, recognized 2 vocabulary term(s) from movie-review training data)

"Recovery was slow but ultimately the outcome was favorable."
  -> predicted: positive (confidence 0.51, recognized 2 vocabulary term(s) from movie-review training data)

"Side effects were mild and the patient tolerated the procedure well."
  -> predicted: positive (confidence 0.55, recognized 3 vocabulary term(s) from movie-review training data)

"The intervention was contraindicated given the patient's prior history."
  -> predicted: positive (confidence 0.53, recognized 2 vocabulary term(s) from movie-review training data)



**Answer in this cell:** Does the model's sentiment classification remain sensible in a medical context? How many vocabulary terms from the medical sentences did the movie-trained model actually recognize? What is the model actually doing when it classifies these sentences — is it detecting sentiment in the clinical sense, or pattern-matching (or effectively guessing) on whatever fragments of its training vocabulary happen to appear?

### Part 3 (Fallback) — Prompting a Generative Model

A real GPT-style model is sensitive to prompt structure in sophisticated ways (few-shot in-context learning, instruction following) that a tiny model cannot replicate. To still illustrate that **prompt structure changes generative output**, even crudely, we build a small Markov-chain text generator from a short built-in corpus and observe how three prompt formats change what it produces.

In [6]:
if not HF_AVAILABLE:
    import random
    random.seed(SEED)

    corpus = (
        "thunderstorms form when warm moist air rises rapidly and cools. "
        "as the air cools water vapor condenses into clouds and releases heat. "
        "this heat causes the air to rise even faster building a tall storm cloud. "
        "strong updrafts and downdrafts inside the cloud create turbulence and static charge. "
        "when the charge separation becomes large enough lightning discharges between the cloud and the ground. "
        "rain forms when water droplets in the cloud grow heavy enough to fall. "
        "briefly explained a thunderstorm needs warm moist air instability and a lifting mechanism."
    )
    words = corpus.split()

    # Build a simple order-1 Markov chain: word -> list of words that followed it in the corpus
    chain = {}
    for w1, w2 in zip(words[:-1], words[1:]):
        chain.setdefault(w1, []).append(w2)

    def generate(seed_words, n_words=25):
        current = seed_words[-1].lower().strip("?:.")
        output = list(seed_words)
        for _ in range(n_words):
            candidates = chain.get(current)
            if not candidates:
                current = random.choice(words)
                continue
            nxt = random.choice(candidates)
            output.append(nxt)
            current = nxt
        return " ".join(output)

    prompts = {
        "Direct question": ["What", "causes", "thunderstorms"],
        "Instruction format": ["Explain", "briefly", "thunderstorms"],
        "Few-shot primed": ["thunderstorms", "form", "when"],
    }

    for style, seed_words in prompts.items():
        print(f"{style}: {generate(seed_words)}\n")


Direct question: What causes thunderstorms form when water droplets in the air rises rapidly and cools. as the air cools water vapor condenses into clouds and cools. as the cloud

Instruction format: Explain briefly thunderstorms form when warm moist air rises rapidly and downdrafts inside the cloud create turbulence and static charge. when water droplets in the air cools water

Few-shot primed: thunderstorms form when warm moist air instability and static charge. when the cloud create turbulence and the ground. rain forms when the charge separation becomes large enough lightning



**Answer in this cell:** Do the three outputs differ in content or style, even with this simple model? What does this crude demonstration suggest about why prompt structure matters more for real, larger language models — and what capabilities would a real GPT-style model bring to this same task that a Markov chain fundamentally cannot?

## Reflection (200-300 words)

"In Part 2, you observed that a model trained on one domain produces outputs when given text from a different domain — but those outputs may be misleading. The model does not know that it is outside its training distribution; it simply processes whatever it is given and produces a prediction.

Think about your own MIPDS application. What is the training distribution of the pre-trained model you have chosen? Where is the language your system will encounter in deployment likely to diverge from that distribution? What would it look like for your system to detect — rather than silently mishandle — a distribution-shifted input?" 

**Your reflection (edit this cell):**

_Write your 200-300 word reflection here._